Generate qiskit metal designs for predicted and reference designs to illustrate that the MLP comes up with alternative designs from SQuADDS DB that still reconstruct the same Hamiltonian parameters well

In [1]:
from pathlib import Path
from parameters import *
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.qlibrary.qubits.transmon_cross import TransmonCross
import pandas as pd
import numpy as np
import random
from squadds import Analyzer, SQuADDS_DB

In [2]:
def percentDiff(x1,x2):
    return (x1 - x2)/abs(x1) * 100

In [3]:
ML_results = pd.read_csv(str(Path(RESULTS_DIR) / "validation/Hamiltonian-based-inverse+surrogate_MLP_results.csv"))
simulation_results = pd.read_csv(str(Path(RESULTS_DIR) / "validation/Hamiltonian-based-inverse+surrogate_Ansys_results.csv"))

In [14]:
ML_results.iloc[76]

ref_qubit_frequency_GHz                          4.665420
ref_anharmonicity_MHz                         -198.548110
ref_connection_pads.readout.claw_length          0.000080
ref_connection_pads.readout.ground_spacing       0.000004
ref_cross_length                                 0.000210
pred_qubit_frequency_GHz                         4.660758
pred_anharmonicity_MHz                        -198.501450
pred_connection_pads.readout.claw_length         0.000107
pred_connection_pads.readout.ground_spacing      0.000004
pred_cross_length                                0.000211
Name: 76, dtype: float64

## choose a random test sample or select sample hand

Figure 13 in paper uses sample 76

In [4]:
sample_to_show = 76 #random.choice(simulation_results.Sample)
print(sample_to_show)
ML_choice = ML_results.iloc[sample_to_show]
sim_choice = simulation_results[simulation_results.Sample == sample_to_show].iloc[0]

76


## create template qubit from SQuADDS

In [5]:
# First, get a design from SQuADDS
db = SQuADDS_DB()
db.select_system("qubit")
db.select_qubit("TransmonCross")
df = db.create_system_df()
analyzer = Analyzer(db)

# Define target parameters for your qubit
target_params = {"qubit_frequency_GHz": 4.2,"anharmonicity_MHz": -200}
# Find the closest design
results = analyzer.find_closest(target_params, num_top=1)

template_device = results.iloc[0]

qubit_options = template_device["design_options"]
qubit_options["pos_x"] = "0um"
qubit_options["pos_y"] = "0um"
qubit_options["connection_pads"]["readout"]["connector_location"] = '0'
qubit_options["orientation"] = "0"

INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/SQuADDS/SQuADDS_DB/0bd24c884ce78504f76deb1887011a7210554482/README.md "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/0bd24c884ce78504f76deb1887011a7210554482/SQuADDS_DB.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/SQuADDS/SQuADDS_DB/SQuADDS/SQuADDS_DB.py "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/datasets/SQuADDS/SQuADDS_DB/revision/0bd24c884ce78504f76deb1887011a7210554482 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/datasets/SQuADDS/SQuADDS_DB/resolve/0bd24c884ce78504f76deb1887011a7210554482/.huggingface.yaml "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://

Generating train split:   0%|          | 0/1934 [00:00<?, ? examples/s]

## function to save source material for Figure 13

In [6]:
import matplotlib.pyplot as plt

def save_metal_design_plot(
    design,
    filename,
    xlim=(-0.67, 0.67),
    ylim=(-0.4, 0.4),
    figsize=(12, 6),
    dpi=300,
):
    poly = design.qgeometry.tables["poly"]

    metal = poly[poly["subtract"] == False]
    etch  = poly[poly["subtract"] == True]

    ground_color = "#B0CCA8"
    gap_color    = "#B0B8C0"
    metal_color  = "#90A88C"
    edge_color   = "#6C7478"

    fig, ax = plt.subplots(figsize=figsize)
    fig.patch.set_facecolor("white")
    ax.set_facecolor(ground_color)

    def draw_geom(geom, facecolor, zorder):
        if geom.geom_type == "Polygon":
            geoms = [geom]
        elif geom.geom_type == "MultiPolygon":
            geoms = list(geom.geoms)
        else:
            return

        for g in geoms:
            x, y = g.exterior.xy
            ax.fill(
                x, y,
                facecolor=facecolor,
                edgecolor=edge_color,
                linewidth=1.2,
                zorder=zorder,
            )

    # grey etched gaps first
    for _, row in etch.iterrows():
        draw_geom(row.geometry, gap_color, zorder=1)

    # green metal on top
    for _, row in metal.iterrows():
        draw_geom(row.geometry, metal_color, zorder=2)

    ax.set_aspect("equal")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.grid(False)
    ax.set_title("")

    fig.savefig(filename, dpi=dpi, bbox_inches="tight", pad_inches=0)
    plt.close(fig)

## reference design

In [8]:
ref_claw_length = str(ML_choice["ref_connection_pads.readout.claw_length"]*10**6)+"um"
ref_ground_spacing = str(ML_choice["ref_connection_pads.readout.ground_spacing"]*10**6)+"um"
ref_cross_length = str(ML_choice["ref_cross_length"]*10**6)+"um"
ref_freq = eval(sim_choice.ref_H_params)["qubit_frequency_GHz"]
ref_alpha = eval(sim_choice.ref_H_params)["anharmonicity_MHz"]

qubit_options["connection_pads"]["readout"]["claw_length"] = ref_claw_length
qubit_options["connection_pads"]["readout"]["ground_spacing"] = ref_ground_spacing
qubit_options["cross_length"] = ref_cross_length

ref_design = designs.DesignPlanar()
ref_gui = MetalGUI(ref_design)

ref_qubit = TransmonCross(ref_design,'ref_qubit',options=qubit_options)

ref_gui.rebuild()
ref_gui.autoscale()

ref_fig = ref_gui.canvas.figure   ## or sometimes .fig depending on version
ref_ax = ref_fig.axes[0]
ref_ax.set_xlim(-.4,.4)
ref_ax.set_ylim(-.4,.4)
ref_file = f'ref_design_{sample_to_show}.pdf'
ref_gui.main_window.force_close = True
ref_gui.main_window.close()

print("reference design parameters:")
print("claw length = ",ref_claw_length)
print("ground spacing = ",ref_ground_spacing)
print("cross length = ",ref_cross_length)
print()
print("reference Hamiltonian parameters:")
print("frequency = ",ref_freq)
print("anharmonicity = ",ref_alpha)

 /Users/firasabouzahr/miniforge3/envs/qubit_design/lib/python3.11/site-packages/qiskit_metal/qgeometries/qgeometries_handler.py: 528
 /Users/firasabouzahr/miniforge3/envs/qubit_design/lib/python3.11/site-packages/qiskit_metal/renderers/renderer_mpl/mpl_renderer.py: 281


reference design parameters:
claw length =  80.0um
ground spacing =  4.1um
cross length =  210.0um

reference Hamiltonian parameters:
frequency =  4.6654196
anharmonicity =  -198.54811


In [9]:
save_metal_design_plot(ref_design,ref_file)

## predicted design

In [10]:
pred_claw_length = str(ML_choice["pred_connection_pads.readout.claw_length"]*10**6)+"um"
pred_ground_spacing = str(ML_choice["pred_connection_pads.readout.ground_spacing"]*10**6)+"um"
pred_cross_length = str(ML_choice["pred_cross_length"]*10**6)+"um"
pred_freq = eval(sim_choice.pred_H_params)["qubit_frequency_GHz"]
pred_alpha = eval(sim_choice.pred_H_params)["anharmonicity_MHz"]

qubit_options["connection_pads"]["readout"]["claw_length"] = pred_claw_length
qubit_options["connection_pads"]["readout"]["ground_spacing"] = pred_ground_spacing
qubit_options["cross_length"] = pred_cross_length

pred_design = designs.DesignPlanar()
pred_gui = MetalGUI(pred_design)

pred_qubit = TransmonCross(pred_design,'pred_qubit',options=qubit_options)

pred_gui.rebuild()
pred_gui.autoscale()

pred_fig = pred_gui.canvas.figure   ## or sometimes .fig depending on version
pred_ax = pred_fig.axes[0]
pred_ax.set_xlim(-.4,.4)
pred_ax.set_ylim(-.4,.4)
pred_file = f'pred_design_{sample_to_show}.pdf'
pred_gui.main_window.force_close = True
pred_gui.main_window.close()

print("predicted design parameters:")
print("claw length = ",pred_claw_length)
print("ground spacing = ",pred_ground_spacing)
print("cross length = ",pred_cross_length)
print()
print("predicted Hamiltonian parameters:")
print("frequency = ",pred_freq)
print("anharmonicity = ",pred_alpha)

 /Users/firasabouzahr/miniforge3/envs/qubit_design/lib/python3.11/site-packages/qiskit_metal/qgeometries/qgeometries_handler.py: 528
 /Users/firasabouzahr/miniforge3/envs/qubit_design/lib/python3.11/site-packages/qiskit_metal/renderers/renderer_mpl/mpl_renderer.py: 281


predicted design parameters:
claw length =  107.071995um
ground spacing =  4.493991um
cross length =  210.85443999999998um

predicted Hamiltonian parameters:
frequency =  4.653004487487429
anharmonicity =  -197.3847594007285


In [ ]:
save_metal_design_plot(pred_design,pred_file)

In [11]:
print("percent difference between reference and predicted design parameters")
print("claw length % difference = ",str(percentDiff(float(ref_claw_length[:-2]),float(pred_claw_length[:-2])))+"%")
print("ground spacing % difference = ",str(percentDiff(float(ref_ground_spacing[:-2]),float(pred_ground_spacing[:-2])))+"%")
print("cross length % difference = ",str(percentDiff(float(ref_cross_length[:-2]),float(pred_cross_length[:-2])))+"%")
print()
print("percent difference between reference and predicted Hamiltonian parameters")
print("frequency % difference = ",str(percentDiff(ref_freq,pred_freq))+"%")
print("anharmonicity % difference = ",str(percentDiff(ref_alpha,pred_alpha))+"%")

percent difference between reference and predicted design parameters
claw length % difference =  -33.839993750000005%
ground spacing % difference =  -9.60953658536587%
cross length % difference =  -0.4068761904761822%

percent difference between reference and predicted Hamiltonian parameters
frequency % difference =  0.26610923726068836%
anharmonicity % difference =  -0.5859288206125436%
